In [1]:
import re
import pandas as pd
from paddleocr import PaddleOCR 
import matplotlib.pyplot as plt
import os

# Initialize OCR
ocr = PaddleOCR(use_angle_cls=False, lang='en')

# empty DataFrame
df_reciept_records = pd.DataFrame(columns=["File Name","Store Name", "Address", "Date", "Items", "Total", "Raw Text"])

C:\Users\Workspace\AppData\Local\Temp\ipykernel_11136\898534663.py:8: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=False, lang='en')
D:\Workspace\Anaconda\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in C:\Users\Workspace\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None)
The model(UVDoc) is not supported to run in MKLDNN mode! Using `paddle` instead!
Using official model (UVDoc), the model files will be automatically downloaded and saved in C:\Users\Workspace\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv5_server_det', None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in C:\Users\Workspace\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv5_server_rec', None)
Using official model (PP-OCRv5_server_rec), the model files will be automatically downloaded and saved in C:\Users\Workspace\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

#SCANNING IMAGES USING OCR

In [3]:
def clean_total_amount(amount):
    if ',' in amount:
        amount.replace(',', '')
    if(amount[0] == '\n'):
        amount =  amount[1:]
    return float(amount)
    
def process_receipt(img_path, i):
    print("scanning...")
    result = ocr.predict(img_path)
    lines = result[0]['rec_texts']

    full_text = "\n".join(lines)

    # Store Name (first line with only caps & symbols, no lowercase letters)
    store_name = next((l for l in lines if re.match(r'^[^a-z]+$', l.strip())), "Unknown")

    # Address (first 2–3 lines after store name that contain numbers + street keywords)
    address_lines = []
    address_keywords = ["JALAN", "ROAD", "STREET", "TAMPOI", "AVENUE", "BAHRU", "ALAM", "CITY", "TOWN"]
    for line in lines:
        if any(k in line.upper() for k in address_keywords):
            address_lines.append(line)
    address = ", ".join(address_lines) if address_lines else "Unknown"

    # Date
    date_match = re.search(r"\d{2}[-/]\d{2}[-/]\d{4}|\d{4}[-/]\d{2}[-/]\d{2}", full_text)
    date = date_match.group(0) if date_match else "Unknown"

    # Total Amount
    total_match = re.search(r"(\$|RM)?\s*\d+\.\d{2}", full_text)
    total_amount = total_match.group(0) if total_match else "Unknown"
    total_amount = clean_total_amount(total_amount)
    # Items (lines containing quantity & price)
    items = []
    item_pattern = re.compile(r".+\s+\d+(\.\d{2})?\s+(\$|RM)?\d+\.\d{2}")
    for line in lines:
        if item_pattern.match(line):
            items.append(line)
    items_text = "; ".join(items) if items else "Unknown"

    # Append to DataFrame
    global df_reciept_records 
    df_reciept_records  = pd.concat([df_reciept_records, pd.DataFrame([{
        "File Name" : img_path,
        "Store Name": store_name,
        "Address": address,
        "Date": date,
        "Items": items_text,
        "Total": total_amount,
        "Raw Text": full_text
    }])], ignore_index=True)
    print(f"scanned image no.{i}\n")

# use this function to scan all the recipts in a folder
def scan_reciepts(folder):
    i = 0
    for file in os.listdir(folder):# file is the path of img 
        print(f"remaining files to scan: {len(os.listdir(folder)) - i}")
        process_receipt(os.path.join(folder,file) , i)
        i = i +1

In [ ]:
img_folder = "model_testing_images" 
scan_reciepts(img_folder)
#scanning takes avearage 1 min 40 secs per image
        

remaining files to scan: 12
scanning...


C:\Users\Workspace\AppData\Local\Temp\ipykernel_11136\170278602.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_reciept_records  = pd.concat([df_reciept_records, pd.DataFrame([{


scanned image no.0

remaining files to scan: 11
scanning...
scanned image no.1

remaining files to scan: 10
scanning...
scanned image no.2

remaining files to scan: 9
scanning...
scanned image no.3

remaining files to scan: 8
scanning...
scanned image no.4

remaining files to scan: 7
scanning...
scanned image no.5

remaining files to scan: 6
scanning...
scanned image no.6

remaining files to scan: 5
scanning...
scanned image no.7

remaining files to scan: 4
scanning...


2️⃣ Visualization

In [ ]:
import matplotlib.pyplot as plt

def visualize_data(df):
    # Convert Total Amount to float for plotting
    df["Total"] = df["Total"].replace(r"[^\d.]", "", regex=True).astype(float)

    # Spending over time
    plt.figure(figsize=(8, 5))
    df.groupby("Date")["Total"].sum().plot(kind="bar")
    plt.title("Total Spending by Date")
    plt.xlabel("Date")
    plt.ylabel("Amount")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("spending_by_date.png")
    plt.close()

    # Spending by Store
    plt.figure(figsize=(8, 5))
    df.groupby("Store Name")["Total"].sum().plot(kind="bar")
    plt.title("Total Spending by Store")
    plt.xlabel("company")
    plt.ylabel("Amount")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("spending_by_store.png")
    plt.close()

visualize_data(df_reciept_records)

PDF Report Generation

In [ ]:
def _sanitize_for_paragraph(s, max_chars=200):
    import re
    if s is None: return ""
    s = str(s).replace('\r\n','\n').replace('\r','\n')
    s = re.sub(r'<br\s*/?>', '\n', s, flags=re.IGNORECASE)
    s = s.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
    if max_chars and len(s) > max_chars:
        s = s[:max_chars] + "\n... [truncated]"
    s = s.replace('\n','<br/>')
    return s

from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
def create_structured_pdf_only(df, filename="Receipt_Report.pdf",
                               drop_cols=["Raw Text", "FileName"],
                               truncate_cols=["Store Name", "Address"], max_chars=200,
                               chunk_size=30):
    import pandas as pd
    df2 = df.copy()
    df2 = df2.drop(columns=drop_cols, errors='ignore')
    for c in (truncate_cols or []):
        if c in df2.columns:
            df2[c] = df2[c].astype(str).apply(lambda s: s if len(s) <= max_chars else s[:max_chars] + " ...")
    if "Total" in df2.columns:
        df2["_Total_num"] = pd.to_numeric(df2["Total"].astype(str).replace(r"[^\d\.]", "", regex=True), errors='coerce')
    doc = SimpleDocTemplate(filename, pagesize=A4, leftMargin=20, rightMargin=20, topMargin=20, bottomMargin=20)
    styles = getSampleStyleSheet()
    small = ParagraphStyle('small', parent=styles['Normal'], fontSize=8, leading=10)
    title_style = styles['Title']
    elements = []
    elements.append(Paragraph("Receipt OCR - Structured Report", title_style))
    elements.append(Spacer(1, 8))
    if "_Total_num" in df2.columns:
        total_spent = df2["_Total_num"].sum(skipna=True)
        elements.append(Paragraph(f"Total Receipts: {len(df2)}", small))
        elements.append(Paragraph(f"Total Spending: ${total_spent:.2f}", small))
        elements.append(Spacer(1, 8))
    # Optional images (try/except)
    try:
        elements.append(Paragraph("Spending Over Time", styles['Heading2']))
        elements.append(Image("spending_by_date.png", width=400, height=250))
        elements.append(Spacer(1, 8))
    except Exception: pass
    try:
        elements.append(Paragraph("Spending by Store", styles['Heading2']))
        elements.append(Image("spending_by_store.png", width=400, height=250))
        elements.append(Spacer(1, 8))
    except Exception: pass
    cols = list(df2.columns)
    usable_width = A4[0] - doc.leftMargin - doc.rightMargin
    for start in range(0, len(df2), chunk_size):
        chunk = df2.iloc[start:start+chunk_size]
        table_data = [[Paragraph(str(col), small) for col in cols]]
        for _, row in chunk.iterrows():
            row_cells = []
            for c in cols:
                val = row[c]
                safe_html = _sanitize_for_paragraph(val, max_chars=max_chars)
                row_cells.append(Paragraph(safe_html, small))
            table_data.append(row_cells)
        col_count = len(cols) if len(cols) > 0 else 1
        col_widths = [usable_width / col_count] * col_count
        tbl = Table(table_data, colWidths=col_widths, repeatRows=1)
        tbl.setStyle(TableStyle([
            ('GRID', (0,0), (-1,-1), 0.25, colors.grey),
            ('VALIGN', (0,0), (-1,-1), 'TOP'),
            ('FONTSIZE', (0,0), (-1,-1), 8),
        ]))
        elements.append(tbl)
        elements.append(PageBreak())
    doc.build(elements)
    print(f"Created structured PDF: {filename}")


In [ ]:
create_structured_pdf_only(df_reciept_records)